# Model Configuration Decision Study — Condition A

## Warm-up example

Imagine the task says:

- quality must be between `0.80` and `0.92`
- training fraction must be between `0.40` and `0.70`
- review budget must be at most `0.40`

| run_id | quality_score | training_fraction | review_budget | corruption_level | Result |
|---:|---:|---:|---:|---|---|
| 101 | 0.85 | 0.55 | 0.30 | mild | valid |
| 102 | 0.94 | 0.55 | 0.30 | mild | invalid: quality is too high |
| 103 | 0.85 | 0.80 | 0.30 | mild | invalid: training fraction is too high |

Use the same logic in the real tasks below.

In [ ]:
#@title Start session (Please, do not modify this cell just run it){ display-mode: "form" }
# Run this cell once at the beginning.

from datetime import datetime
from zoneinfo import ZoneInfo
import secrets
from pathlib import Path
import subprocess
import sys
import urllib.request

TIMEZONE = "Europe/Warsaw"

# Generate stable IDs for this notebook session.
# They will not change if this cell is re-run in the same active kernel.
if "PARTICIPANT_ID" not in globals() or not str(globals().get("PARTICIPANT_ID", "")).strip():
    PARTICIPANT_ID = f"P-{secrets.token_hex(3).upper()}"

if "SESSION_ID" not in globals() or not str(globals().get("SESSION_ID", "")).strip():
    SESSION_ID = datetime.now(ZoneInfo(TIMEZONE)).strftime("%Y%m%d_%H%M%S")

QUESTIONNAIRE_ID = PARTICIPANT_ID

# Study metadata. Do not edit unless the facilitator asks you to.
STUDY_CONDITION = "condition_a"
NOTEBOOK_VERSION = "2026-04-30-condition-a-v5"

# Fixed seeds: all participants see the same table order and the same Task 3 base.
STUDY_RANDOM_SEED = 42
WHAT_IF_BASE_RANDOM_SEED = STUDY_RANDOM_SEED + 300

print("Session started.")

REPO = "sabri-manai/user-study-cf-hpo-xai"
REQUIREMENTS_PATH = Path("requirements.txt")
REQUIREMENTS_URL = f"https://raw.githubusercontent.com/{REPO}/main/requirements.txt"

if not REQUIREMENTS_PATH.exists():
    urllib.request.urlretrieve(REQUIREMENTS_URL, REQUIREMENTS_PATH)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    str(REQUIREMENTS_PATH),
])

print("Requirements installed.")


from pathlib import Path
import json
import re
import time
import urllib.request

import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_STARTED_AT = time.time()
ATTEMPT_LOG = []
CHECK_HISTORY = []

OUTPUT_DIR = Path("student_condition_a_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = Path("surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv")
DATA_URL = "https://raw.githubusercontent.com/sabri-manai/user-study-cf-hpo-xai/main/surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv"

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

TARGET = "value"

RAW_DISPLAY_COLUMNS = [
    "run_id",
    "value",
    "params_soft_train_fraction",
    "params_soft_review_budget",
    "params_soft_corruption_level",
    "params_backbone",
    "params_batch_size",
    "params_image_size_key",
    "params_layers_key",
    "params_num_neighbors",
    "params_reduction",
]

COLUMN_LABELS = {
    "value": "quality_score",
    "params_soft_train_fraction": "training_fraction",
    "params_soft_review_budget": "review_budget",
    "params_soft_corruption_level": "corruption_level",
    "params_backbone": "backbone",
    "params_batch_size": "batch_size",
    "params_image_size_key": "image_size",
    "params_layers_key": "layers",
    "params_num_neighbors": "num_neighbors",
    "params_reduction": "reduction",
}

TASKS = {
    "task_1": {
        "quality_range": (0.80, 0.92),
        "train_fraction_range": (0.40, 0.70),
        "review_budget_range": (0.00, 0.40),
        "corruption_allowed": ["none", "mild", "strong"],
    },
    "task_2": {
        "quality_range": (0.78, 0.91),
        "train_fraction_range": (0.20, 0.60),
        "review_budget_range": (0.00, 0.30),
        "corruption_allowed": ["none", "mild"],
    },
}

WHAT_IF_TASK = {
    "task_name": "task_3",
    "review_budget_multiplier": 0.90,
    "quality_tolerance": 0.01,
    "n_valid_options": 4,
    "n_quality_invalid_options": 3,
    "n_budget_invalid_options": 3,
}

CONFIDENCE_LEVELS = {"low", "medium", "high"}

# Load previous HPO runs.
df = pd.read_csv(DATA_PATH).reset_index().rename(columns={"index": "run_id"})

for col in ["value", "params_soft_train_fraction", "params_soft_review_budget"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["params_soft_corruption_level"] = df["params_soft_corruption_level"].astype(str)


def student_view(rows):
    """Return only participant-facing columns, with friendly names."""
    cols = []
    if "display_order" in rows.columns:
        cols.append("display_order")
    cols += RAW_DISPLAY_COLUMNS
    return rows[cols].rename(columns=COLUMN_LABELS)


def safe_token(value, fallback="unknown"):
    value = str(value).strip()
    if not value:
        return fallback
    return re.sub(r"[^A-Za-z0-9_-]+", "_", value)


def now_relative_seconds():
    return round(time.time() - NOTEBOOK_STARTED_AT, 2)


def log_attempt(task_name, run_id, justification, confidence, extra=None):
    """Save every time an answer cell is run."""
    entry = {
        "attempt_index": len(ATTEMPT_LOG) + 1,
        "task_name": task_name,
        "relative_seconds": now_relative_seconds(),
        "run_id": str(run_id).strip() if run_id is not None else "",
        "justification": str(justification).strip(),
        "confidence": str(confidence).strip(),
    }

    if extra:
        entry.update(extra)

    ATTEMPT_LOG.append(entry)
    print(f"Saved attempt {entry['attempt_index']} for {task_name} at {entry['relative_seconds']} seconds.")


print(f"Loaded {len(df)} previous HPO runs.")

print(f"Participant ID: {QUESTIONNAIRE_ID}")

Session started.
Participant ID / Questionnaire ID: P-F54434
Session ID: 20260505_200233


In [ ]:
#@title Show option tables for Task 1 and Task 2 (Please do not modify this cell just run it)
# The displayed order is fixed for all participants, but it is not sorted by quality.

OPTION_TABLES_DISPLAYED_AT = time.time()

OPTION_RUN_IDS = {
    "task_1": [2738, 2565, 2544, 2449, 2541, 2553, 2572, 2549, 1213, 956, 1098, 615],
    "task_2": [2738, 2553, 2565, 2656, 2549, 2541, 2544, 2548, 1317, 716, 1481],
}


def valid_mask_for(task, dataframe=df):
    q_low, q_high = task["quality_range"]
    tf_low, tf_high = task["train_fraction_range"]
    rb_low, rb_high = task["review_budget_range"]
    allowed = task["corruption_allowed"]

    return (
        dataframe["value"].between(q_low, q_high, inclusive="both")
        & dataframe["params_soft_train_fraction"].between(tf_low, tf_high, inclusive="both")
        & dataframe["params_soft_review_budget"].between(rb_low, rb_high, inclusive="both")
        & dataframe["params_soft_corruption_level"].isin(allowed)
    )


def options_for(task_name):
    option_ids = OPTION_RUN_IDS[task_name]
    options = df[df["run_id"].isin(option_ids)].copy()

    task_seed = STUDY_RANDOM_SEED + (1 if task_name == "task_1" else 2)
    options = options.sample(frac=1, random_state=task_seed).reset_index(drop=True)

    options["display_order"] = range(1, len(options) + 1)
    return options


OPTIONS = {name: options_for(name) for name in TASKS}


def displayed_order_records(option_tables):
    records = []
    for task_name, table in option_tables.items():
        for _, row in table[["run_id", "display_order"]].iterrows():
            records.append({
                "task_name": task_name,
                "run_id": int(row["run_id"]),
                "display_order": int(row["display_order"]),
            })
    return records


def get_display_order(task_name, run_id):
    table = OPTIONS[task_name]
    match = table.loc[table["run_id"] == int(run_id), "display_order"]
    if match.empty:
        return None
    return int(match.iloc[0])


def independent_task_violations(task_name, run_id):
    """Return a list of violated constraints for task_1 or task_2."""
    if run_id is None or str(run_id).strip() == "":
        return ["missing run_id"]

    run_id = int(run_id)

    if run_id not in set(OPTIONS[task_name]["run_id"]):
        return ["run_id is not from the displayed option table"]

    row = df.loc[df["run_id"] == run_id].iloc[0]
    task = TASKS[task_name]

    q_low, q_high = task["quality_range"]
    tf_low, tf_high = task["train_fraction_range"]
    rb_low, rb_high = task["review_budget_range"]
    allowed = task["corruption_allowed"]

    violations = []

    if not (q_low <= float(row["value"]) <= q_high):
        violations.append("quality_score outside allowed range")

    if not (tf_low <= float(row["params_soft_train_fraction"]) <= tf_high):
        violations.append("training_fraction outside allowed range")

    if not (rb_low <= float(row["params_soft_review_budget"]) <= rb_high):
        violations.append("review_budget outside allowed range")

    if str(row["params_soft_corruption_level"]) not in allowed:
        violations.append("corruption_level not allowed")

    return violations


def assert_independent_choice_is_valid(task_name, run_id):
    violations = independent_task_violations(task_name, run_id)
    if violations:
        joined = "; ".join(violations)
        raise ValueError(f"{task_name}: selected run_id {run_id} is not valid: {joined}")
    return True


for name, table in OPTIONS.items():
    print(f"\n{name}: mixed option table ({len(table)} rows). Not every row is valid.")
    display(student_view(table))


task_1: mixed option table (12 rows). Not every row is valid.


,display_order,run_id,quality_score,training_fraction,review_budget,corruption_level,backbone,batch_size,image_size,layers,num_neighbors,reduction
0,1,2738,0.907563,0.90,0.340,none,resnet34,16,224,l3,1,mean
1,2,2544,0.906667,0.85,0.345,none,resnet34,16,224,l3,1,mean
2,3,2549,0.906667,0.86,0.345,none,resnet34,16,224,l3,1,mean
3,4,2553,0.906667,0.86,0.345,none,resnet34,16,224,l3,1,mean
4,5,2565,0.906667,0.84,0.345,none,resnet34,16,224,l3,1,mean
5,6,1213,0.859425,0.55,0.375,none,resnet18,8,224,l2_l3,3,mean
6,7,2572,0.906667,0.86,0.345,none,resnet34,16,224,l3,1,mean
7,8,1098,0.843800,0.56,0.370,mild,resnet34,16,256,l2_l3,8,max
8,9,2541,0.906667,0.80,0.345,none,resnet34,16,224,l3,1,mean
9,10,956,0.859351,0.67,0.400,none,resnet50,8,224,l1_l2_l3,12,mean



task_2: mixed option table (11 rows). Not every row is valid.


,display_order,run_id,quality_score,training_fraction,review_budget,corruption_level,backbone,batch_size,image_size,layers,num_neighbors,reduction
0,1,2565,0.906667,0.84,0.345,none,resnet34,16,224,l3,1,mean
1,2,2553,0.906667,0.86,0.345,none,resnet34,16,224,l3,1,mean
2,3,1481,0.806569,0.32,0.285,none,resnet50,8,224,l2_l3,17,max
3,4,2738,0.907563,0.90,0.340,none,resnet34,16,224,l3,1,mean
4,5,716,0.809269,0.39,0.300,mild,resnet18,16,256,l3,11,mean
5,6,2549,0.906667,0.86,0.345,none,resnet34,16,224,l3,1,mean
6,7,2656,0.906667,0.87,0.345,none,resnet34,16,224,l3,1,mean
7,8,2548,0.906667,0.83,0.345,none,resnet34,16,224,l3,1,mean
8,9,1317,0.823985,0.32,0.250,none,resnet34,16,256,l1_l2_l3,16,max
9,10,2541,0.906667,0.80,0.345,none,resnet34,16,224,l3,1,mean


In [ ]:
#@title Task 1 answer
#@markdown ### Type your answer and run this cell.
#@markdown Choose one `run_id` from the displayed Task 1 option table.
#@markdown Also write a short justification and select your confidence.

TASK_1_RUN_ID = ""  #@param {type:"string"}
TASK_1_JUSTIFICATION = ""  #@param {type:"string"}
TASK_1_CONFIDENCE = ""  #@param ["", "low", "medium", "high"]

TASK_1_ANSWERED_AT = time.time()

log_attempt(
    task_name="task_1",
    run_id=TASK_1_RUN_ID,
    justification=TASK_1_JUSTIFICATION,
    confidence=TASK_1_CONFIDENCE,
)

Saved attempt 1 for task_1 at 509.82 seconds.


In [ ]:
#@title Task 2 answer
#@markdown ### Type your answer and run this cell.
#@markdown Choose one `run_id` from the displayed Task 2 option table.
#@markdown Also write a short justification and select your confidence.


TASK_2_RUN_ID = "1317"  #@param {type:"string"}
TASK_2_JUSTIFICATION = "valid soft constraints good quality"  #@param {type:"string"}
TASK_2_CONFIDENCE = "medium"  #@param ["", "low", "medium", "high"]

TASK_2_ANSWERED_AT = time.time()

log_attempt(
    task_name="task_2",
    run_id=TASK_2_RUN_ID,
    justification=TASK_2_JUSTIFICATION,
    confidence=TASK_2_CONFIDENCE,
)

Saved attempt 2 for task_2 at 722.53 seconds.


In [ ]:
#@title Build and display the Task 3 what-if table, it could take a few minutes to complete.
#@markdown ### Please do not modify this cell just run it!

TASK_3_TABLE_GENERATED_AT = time.time()


def annotate_what_if_candidates(base_run_id):
    """Annotate all possible Task 3 alternatives relative to a base configuration."""
    base = df.loc[df["run_id"] == int(base_run_id)].iloc[0]

    base_value = float(base["value"])
    base_review_budget = float(base["params_soft_review_budget"])

    minimum_acceptable_value = base_value - WHAT_IF_TASK["quality_tolerance"]
    required_max_review_budget = base_review_budget * WHAT_IF_TASK["review_budget_multiplier"]

    candidates = df.loc[df["run_id"] != int(base_run_id)].copy()

    candidates["same_performance_ok"] = candidates["value"] >= minimum_acceptable_value
    candidates["review_budget_10pct_lower_ok"] = (
        candidates["params_soft_review_budget"] <= required_max_review_budget
    )
    candidates["different_from_base_ok"] = candidates["run_id"] != int(base_run_id)

    candidates["valid_what_if"] = (
        candidates["same_performance_ok"]
        & candidates["review_budget_10pct_lower_ok"]
        & candidates["different_from_base_ok"]
    )

    def candidate_type(row):
        if row["valid_what_if"]:
            return "valid"
        if row["review_budget_10pct_lower_ok"] and not row["same_performance_ok"]:
            return "invalid_quality_drop"
        if row["same_performance_ok"] and not row["review_budget_10pct_lower_ok"]:
            return "invalid_budget_reduction"
        return "invalid_both"

    candidates["candidate_type"] = candidates.apply(candidate_type, axis=1)

    candidates["quality_drop_from_base"] = base_value - candidates["value"]
    candidates["review_budget_reduction_absolute"] = (
        base_review_budget - candidates["params_soft_review_budget"]
    )
    candidates["review_budget_reduction_percent"] = (
        candidates["review_budget_reduction_absolute"] / base_review_budget
        if base_review_budget
        else np.nan
    )

    candidates["base_run_id"] = int(base_run_id)
    candidates["base_quality_score"] = base_value
    candidates["base_review_budget"] = base_review_budget
    candidates["minimum_acceptable_quality"] = minimum_acceptable_value
    candidates["required_max_review_budget"] = required_max_review_budget

    return candidates


def choose_fixed_random_what_if_base():
    """
    Choose one fixed random base configuration that has enough valid and invalid alternatives.
    This makes Task 3 comparable across participants.
    """
    eligible_base_ids = []

    for run_id in df["run_id"].tolist():
        candidates = annotate_what_if_candidates(run_id)

        n_valid = int((candidates["candidate_type"] == "valid").sum())
        n_quality_invalid = int((candidates["candidate_type"] == "invalid_quality_drop").sum())
        n_budget_invalid = int((candidates["candidate_type"] == "invalid_budget_reduction").sum())

        if (
            n_valid >= WHAT_IF_TASK["n_valid_options"]
            and n_quality_invalid >= WHAT_IF_TASK["n_quality_invalid_options"]
            and n_budget_invalid >= WHAT_IF_TASK["n_budget_invalid_options"]
        ):
            eligible_base_ids.append(int(run_id))

    if not eligible_base_ids:
        raise ValueError("No base configuration has enough valid and invalid what-if alternatives.")

    rng = np.random.default_rng(WHAT_IF_BASE_RANDOM_SEED)
    selected_base_id = int(rng.choice(sorted(eligible_base_ids)))

    return selected_base_id


WHAT_IF_BASE_RUN_ID = choose_fixed_random_what_if_base()
WHAT_IF_BASE_ROW = df.loc[df["run_id"] == WHAT_IF_BASE_RUN_ID].iloc[0]

WHAT_IF_CANDIDATES = annotate_what_if_candidates(WHAT_IF_BASE_RUN_ID)

valid_options = (
    WHAT_IF_CANDIDATES[WHAT_IF_CANDIDATES["candidate_type"] == "valid"]
    .sample(n=WHAT_IF_TASK["n_valid_options"], random_state=STUDY_RANDOM_SEED + 31)
)

quality_invalid_options = (
    WHAT_IF_CANDIDATES[WHAT_IF_CANDIDATES["candidate_type"] == "invalid_quality_drop"]
    .sample(n=WHAT_IF_TASK["n_quality_invalid_options"], random_state=STUDY_RANDOM_SEED + 32)
)

budget_invalid_options = (
    WHAT_IF_CANDIDATES[WHAT_IF_CANDIDATES["candidate_type"] == "invalid_budget_reduction"]
    .sample(n=WHAT_IF_TASK["n_budget_invalid_options"], random_state=STUDY_RANDOM_SEED + 33)
)

WHAT_IF_OPTIONS = pd.concat(
    [valid_options, quality_invalid_options, budget_invalid_options],
    ignore_index=True,
)

# Fixed shuffled display order.
WHAT_IF_OPTIONS = WHAT_IF_OPTIONS.sample(
    frac=1,
    random_state=STUDY_RANDOM_SEED + 3,
).reset_index(drop=True)

WHAT_IF_OPTIONS["display_order"] = range(1, len(WHAT_IF_OPTIONS) + 1)

base_value = float(WHAT_IF_BASE_ROW["value"])
base_review_budget = float(WHAT_IF_BASE_ROW["params_soft_review_budget"])
minimum_acceptable_value = base_value - WHAT_IF_TASK["quality_tolerance"]
required_max_review_budget = base_review_budget * WHAT_IF_TASK["review_budget_multiplier"]

print("The table below is mixed. Some alternatives are valid and some are not.")

display(student_view(WHAT_IF_OPTIONS))

Task 3 base configuration


,run_id,quality_score,training_fraction,review_budget,corruption_level,backbone,batch_size,image_size,layers,num_neighbors,reduction
1889,1889,0.84984,0.95,0.375,strong,resnet34,16,256,l3,2,max


Task 3 what-if requirement
Base run_id: 1889
Base quality: 0.8498
Base review budget: 0.3750
Alternative quality must be >= 0.8398
Alternative review budget must be <= 0.3375

The table below is mixed. Some alternatives are valid and some are not.


,display_order,run_id,quality_score,training_fraction,review_budget,corruption_level,backbone,batch_size,image_size,layers,num_neighbors,reduction
0,1,2433,0.883162,0.87,0.325,none,resnet34,16,224,l3,2,mean
1,2,1040,0.042781,0.74,0.085,none,wide_resnet50_2,16,224,l3,7,mean
2,3,2794,0.854015,0.87,0.285,none,resnet34,16,224,l3,1,mean
3,4,2530,0.900000,0.85,0.345,none,resnet34,16,224,l3,1,mean
4,5,2407,0.897893,0.90,0.365,none,resnet34,16,224,l3,1,mean
5,6,2790,0.459184,0.89,0.105,none,resnet34,16,224,l3,1,mean
6,7,2695,0.390501,0.85,0.090,none,resnet34,16,224,l3,2,mean
7,8,1691,0.845395,1.00,0.355,strong,resnet34,16,256,l3,1,max
8,9,2939,0.882250,0.86,0.310,none,resnet34,16,224,l3,1,mean
9,10,2654,0.893471,0.86,0.325,none,resnet34,16,224,l3,1,mean


In [ ]:
#@title Display the Task 3 base configuration and requirements.
#@markdown ### Please do not modify this cell just run it!

TASK_3_TABLE_GENERATED_AT = time.time()
print("Task 3 base configuration")
display(student_view(pd.DataFrame([WHAT_IF_BASE_ROW])))

print("Task 3 what-if requirement")
print("\nChoose one alternative that:")
print("- keeps the quality score not lower than the base configuration")
print("- reduces the review budget by at least 10%")

In [ ]:
#@title Task 3 answer
#@markdown ### Type your answer and run this cell.
#@markdown Choose only from the displayed task_3 what-if table.
#@markdown Also write a short justification and select your confidence.


TASK_3_RUN_ID = "2939"  #@param {type:"string"}
TASK_3_JUSTIFICATION = "satisfies constraints "  #@param {type:"string"}
TASK_3_CONFIDENCE = "medium"  #@param ["", "low", "medium", "high"]

TASK_3_ANSWERED_AT = time.time()

log_attempt(
    task_name="task_3",
    run_id=TASK_3_RUN_ID,
    justification=TASK_3_JUSTIFICATION,
    confidence=TASK_3_CONFIDENCE,
    extra={
        "base_run_id": int(WHAT_IF_BASE_RUN_ID) if "WHAT_IF_BASE_RUN_ID" in globals() else None,
    },
)

Saved attempt 3 for task_3 at 1081.48 seconds.


In [ ]:
#@title 9. Check and save answers
# Run this cell after all three tasks are complete.

if not str(PARTICIPANT_ID).strip():
    raise ValueError("Please fill PARTICIPANT_ID in the first cell before saving.")

if "WHAT_IF_OPTIONS" not in globals():
    raise ValueError("Generate the Task 3 what-if table before checking answers.")

if "ATTEMPT_LOG" not in globals():
    ATTEMPT_LOG = []

if "CHECK_HISTORY" not in globals():
    CHECK_HISTORY = []


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def to_jsonable(value):
    """Convert NumPy/Pandas values into JSON-safe Python values."""
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_jsonable(v) for v in value]
    if isinstance(value, tuple):
        return [to_jsonable(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return value


def validate_confidence(task_name, confidence):
    confidence = str(confidence).strip().lower()
    if confidence not in CONFIDENCE_LEVELS:
        raise ValueError(f"{task_name}: confidence must be one of {sorted(CONFIDENCE_LEVELS)}")
    return confidence


def validate_justification(task_name, justification):
    justification = str(justification).strip()
    if len(justification) < 10:
        raise ValueError(f"{task_name}: please write a short justification")
    return justification


def get_final_display_order(task_name, run_id):
    if task_name in OPTIONS:
        table = OPTIONS[task_name]
    elif task_name == "task_3":
        table = WHAT_IF_OPTIONS
    else:
        return None

    match = table.loc[table["run_id"] == int(run_id), "display_order"]
    if match.empty:
        return None
    return int(match.iloc[0])


def compute_rank_fields(table, run_id):
    temp = table.copy()

    temp["quality_rank_desc"] = temp["value"].rank(
        method="min",
        ascending=False,
    ).astype(int)

    temp["review_budget_rank_asc"] = temp["params_soft_review_budget"].rank(
        method="min",
        ascending=True,
    ).astype(int)

    row = temp.loc[temp["run_id"] == int(run_id)]

    if row.empty:
        return {
            "quality_rank_within_displayed_options": None,
            "review_budget_rank_within_displayed_options": None,
        }

    return {
        "quality_rank_within_displayed_options": int(row["quality_rank_desc"].iloc[0]),
        "review_budget_rank_within_displayed_options": int(row["review_budget_rank_asc"].iloc[0]),
    }


def compute_constraint_margins(row, task):
    q_low, q_high = task["quality_range"]
    tf_low, tf_high = task["train_fraction_range"]
    rb_low, rb_high = task["review_budget_range"]

    value = float(row["value"])
    train_fraction = float(row["params_soft_train_fraction"])
    review_budget = float(row["params_soft_review_budget"])

    return {
        "quality_margin_lower": value - q_low,
        "quality_margin_upper": q_high - value,
        "train_fraction_margin_lower": train_fraction - tf_low,
        "train_fraction_margin_upper": tf_high - train_fraction,
        "review_budget_margin_lower": review_budget - rb_low,
        "review_budget_margin_upper": rb_high - review_budget,
    }


def get_attempt_stats(task_name):
    """Summarise all answer-cell attempts for a task."""
    attempts = [a for a in ATTEMPT_LOG if a.get("task_name") == task_name]

    if not attempts:
        return {
            f"{task_name}_attempt_count": 0,
            f"{task_name}_first_attempt_relative_seconds": None,
            f"{task_name}_last_attempt_relative_seconds": None,
        }

    attempt_times = [
        a.get("relative_seconds")
        for a in attempts
        if a.get("relative_seconds") is not None
    ]

    return {
        f"{task_name}_attempt_count": len(attempts),
        f"{task_name}_first_attempt_relative_seconds": min(attempt_times) if attempt_times else None,
        f"{task_name}_last_attempt_relative_seconds": max(attempt_times) if attempt_times else None,
    }


# ---------------------------------------------------------------------
# Task validation
# ---------------------------------------------------------------------

def check_independent_answer(task_name, answer):
    """Check Task 1 or Task 2."""
    run_id = answer.get("run_id")

    if run_id is None or str(run_id).strip() == "":
        raise ValueError(f"{task_name}: please enter a run_id")

    run_id = int(run_id)

    if run_id not in set(OPTIONS[task_name]["run_id"]):
        raise ValueError(f"{task_name}: choose a run_id from the displayed option table only")

    row = df.loc[df["run_id"] == run_id].iloc[0]
    task = TASKS[task_name]

    q_low, q_high = task["quality_range"]
    tf_low, tf_high = task["train_fraction_range"]
    rb_low, rb_high = task["review_budget_range"]
    allowed = task["corruption_allowed"]

    checks = {
        "quality_ok": q_low <= float(row["value"]) <= q_high,
        "train_fraction_ok": tf_low <= float(row["params_soft_train_fraction"]) <= tf_high,
        "review_budget_ok": rb_low <= float(row["params_soft_review_budget"]) <= rb_high,
        "corruption_ok": str(row["params_soft_corruption_level"]) in allowed,
    }

    valid = all(checks.values())

    result = row[RAW_DISPLAY_COLUMNS].to_dict()
    result["task_name"] = task_name
    result["display_order"] = get_final_display_order(task_name, run_id)
    result["shown_in_option_table"] = True

    result.update(compute_rank_fields(OPTIONS[task_name], run_id))
    result.update(compute_constraint_margins(row, task))
    result.update(checks)

    # Not applicable to independent tasks.
    result["base_run_id"] = np.nan
    result["base_quality_score"] = np.nan
    result["base_review_budget"] = np.nan
    result["minimum_acceptable_quality"] = np.nan
    result["required_max_review_budget"] = np.nan
    result["same_performance_ok"] = np.nan
    result["review_budget_10pct_lower_ok"] = np.nan
    result["different_from_base_ok"] = np.nan
    result["quality_drop_from_base"] = np.nan
    result["review_budget_reduction_absolute"] = np.nan
    result["review_budget_reduction_percent"] = np.nan
    result["candidate_type"] = np.nan
    result["valid_what_if"] = np.nan

    result["valid_selection"] = bool(valid)
    result["n_violations"] = int(sum(not ok for ok in checks.values()))
    result["justification"] = validate_justification(task_name, answer.get("justification", ""))
    result["confidence"] = validate_confidence(task_name, answer.get("confidence", ""))

    return result


def check_what_if_answer(answer):
    """Check Task 3 mixed what-if alternative."""
    task_name = "task_3"
    run_id = answer.get("run_id")

    if run_id is None or str(run_id).strip() == "":
        raise ValueError("task_3: please enter a run_id")

    run_id = int(run_id)

    if run_id not in set(WHAT_IF_OPTIONS["run_id"]):
        raise ValueError("task_3: choose a run_id from the displayed what-if table only")

    row = df.loc[df["run_id"] == run_id].iloc[0]
    base = df.loc[df["run_id"] == int(WHAT_IF_BASE_RUN_ID)].iloc[0]

    base_value = float(base["value"])
    base_review_budget = float(base["params_soft_review_budget"])

    selected_value = float(row["value"])
    selected_review_budget = float(row["params_soft_review_budget"])

    minimum_acceptable_value = base_value - WHAT_IF_TASK["quality_tolerance"]
    required_max_review_budget = base_review_budget * WHAT_IF_TASK["review_budget_multiplier"]

    quality_drop = base_value - selected_value
    review_budget_reduction = base_review_budget - selected_review_budget
    review_budget_reduction_pct = (
        review_budget_reduction / base_review_budget
        if base_review_budget
        else np.nan
    )

    checks = {
        "same_performance_ok": selected_value >= minimum_acceptable_value,
        "review_budget_10pct_lower_ok": selected_review_budget <= required_max_review_budget,
        "different_from_base_ok": run_id != int(WHAT_IF_BASE_RUN_ID),
    }

    # Task 3 table is intentionally mixed, so the selected answer must be checked.
    valid = all(checks.values())

    result = row[RAW_DISPLAY_COLUMNS].to_dict()
    result["task_name"] = "task_3"

    result["base_run_id"] = int(WHAT_IF_BASE_RUN_ID)
    result["base_quality_score"] = base_value
    result["base_review_budget"] = base_review_budget
    result["minimum_acceptable_quality"] = minimum_acceptable_value
    result["required_max_review_budget"] = required_max_review_budget

    result["display_order"] = get_final_display_order("task_3", run_id)
    result["shown_in_option_table"] = True

    result.update(compute_rank_fields(WHAT_IF_OPTIONS, run_id))

    # Independent-task constraints are not applicable to Task 3.
    result["quality_ok"] = np.nan
    result["train_fraction_ok"] = np.nan
    result["review_budget_ok"] = np.nan
    result["corruption_ok"] = np.nan

    # Margins relative to the what-if thresholds.
    result["quality_margin_lower"] = selected_value - minimum_acceptable_value
    result["quality_margin_upper"] = np.nan
    result["train_fraction_margin_lower"] = np.nan
    result["train_fraction_margin_upper"] = np.nan
    result["review_budget_margin_lower"] = np.nan
    result["review_budget_margin_upper"] = required_max_review_budget - selected_review_budget

    result.update(checks)

    result["quality_drop_from_base"] = quality_drop
    result["review_budget_reduction_absolute"] = review_budget_reduction
    result["review_budget_reduction_percent"] = review_budget_reduction_pct

    option_match = WHAT_IF_OPTIONS.loc[WHAT_IF_OPTIONS["run_id"] == run_id]

    if not option_match.empty:
        result["candidate_type"] = (
            option_match["candidate_type"].iloc[0]
            if "candidate_type" in option_match.columns
            else np.nan
        )
        result["valid_what_if"] = (
            bool(option_match["valid_what_if"].iloc[0])
            if "valid_what_if" in option_match.columns
            else bool(valid)
        )
    else:
        result["candidate_type"] = np.nan
        result["valid_what_if"] = bool(valid)

    result["valid_selection"] = bool(valid)
    result["n_violations"] = int(sum(not ok for ok in checks.values()))
    result["justification"] = validate_justification("task_3", answer.get("justification", ""))
    result["confidence"] = validate_confidence("task_3", answer.get("confidence", ""))

    return result


# ---------------------------------------------------------------------
# Collect answer snapshots
# ---------------------------------------------------------------------

ANSWERS = {
    "task_1": {
        "run_id": TASK_1_RUN_ID,
        "justification": TASK_1_JUSTIFICATION,
        "confidence": TASK_1_CONFIDENCE,
        "answered_at": globals().get("TASK_1_ANSWERED_AT", None),
    },
    "task_2": {
        "run_id": TASK_2_RUN_ID,
        "justification": TASK_2_JUSTIFICATION,
        "confidence": TASK_2_CONFIDENCE,
        "answered_at": globals().get("TASK_2_ANSWERED_AT", None),
    },
    "task_3": {
        "run_id": TASK_3_RUN_ID,
        "justification": TASK_3_JUSTIFICATION,
        "confidence": TASK_3_CONFIDENCE,
        "answered_at": globals().get("TASK_3_ANSWERED_AT", None),
    },
}

CHECK_RUN_COUNT = globals().get("CHECK_RUN_COUNT", 0) + 1
CHECKED_AT = time.time()

SESSION_ID_FINAL = str(SESSION_ID).strip()
participant_token = safe_token(PARTICIPANT_ID, "participant")
session_token = safe_token(SESSION_ID_FINAL, "session")

QUESTIONNAIRE_ID = PARTICIPANT_ID


# ---------------------------------------------------------------------
# Run checks
# ---------------------------------------------------------------------

records = []
records.append(check_independent_answer("task_1", ANSWERS["task_1"]))
records.append(check_independent_answer("task_2", ANSWERS["task_2"]))
records.append(check_what_if_answer(ANSWERS["task_3"]))

results = pd.DataFrame(records)


# ---------------------------------------------------------------------
# Display results
# ---------------------------------------------------------------------

summary_columns = [
    "task_name",
    "display_order",
    "run_id",
    "value",
    "params_soft_train_fraction",
    "params_soft_review_budget",
    "params_soft_corruption_level",
    "shown_in_option_table",
    "quality_rank_within_displayed_options",
    "review_budget_rank_within_displayed_options",
    "quality_ok",
    "train_fraction_ok",
    "review_budget_ok",
    "corruption_ok",
    "same_performance_ok",
    "review_budget_10pct_lower_ok",
    "different_from_base_ok",
    "quality_margin_lower",
    "quality_margin_upper",
    "train_fraction_margin_lower",
    "train_fraction_margin_upper",
    "review_budget_margin_lower",
    "review_budget_margin_upper",
    "quality_drop_from_base",
    "review_budget_reduction_absolute",
    "review_budget_reduction_percent",
    "candidate_type",
    "valid_what_if",
    "valid_selection",
    "n_violations",
    "confidence",
    "justification",
]

for col in summary_columns:
    if col not in results.columns:
        results[col] = np.nan

results_display = results[summary_columns].rename(columns=COLUMN_LABELS)
display(results_display)

if results["valid_selection"].all():
    print("All selections satisfy the task constraints.")
else:
    print("At least one selection violates the task constraints. Use the check columns above and revise the relevant answer.")


# ---------------------------------------------------------------------
# Timing
# ---------------------------------------------------------------------

task_1_start = globals().get("OPTION_TABLES_DISPLAYED_AT", NOTEBOOK_STARTED_AT)
task_2_start = globals().get("TASK_1_ANSWERED_AT", globals().get("OPTION_TABLES_DISPLAYED_AT", NOTEBOOK_STARTED_AT))
task_3_start = globals().get("TASK_3_TABLE_GENERATED_AT", NOTEBOOK_STARTED_AT)

timing = {
    "notebook_total_seconds_at_check": round(CHECKED_AT - NOTEBOOK_STARTED_AT, 2),

    "task_1_started_at_relative_seconds": round(task_1_start - NOTEBOOK_STARTED_AT, 2),
    "task_1_answered_at_relative_seconds": round(ANSWERS["task_1"]["answered_at"] - NOTEBOOK_STARTED_AT, 2) if ANSWERS["task_1"]["answered_at"] else None,
    "task_1_elapsed_seconds": round(ANSWERS["task_1"]["answered_at"] - task_1_start, 2) if ANSWERS["task_1"]["answered_at"] else None,

    "task_2_started_at_relative_seconds": round(task_2_start - NOTEBOOK_STARTED_AT, 2),
    "task_2_answered_at_relative_seconds": round(ANSWERS["task_2"]["answered_at"] - NOTEBOOK_STARTED_AT, 2) if ANSWERS["task_2"]["answered_at"] else None,
    "task_2_elapsed_seconds": round(ANSWERS["task_2"]["answered_at"] - task_2_start, 2) if ANSWERS["task_2"]["answered_at"] else None,

    "task_3_started_at_relative_seconds": round(task_3_start - NOTEBOOK_STARTED_AT, 2),
    "task_3_answered_at_relative_seconds": round(ANSWERS["task_3"]["answered_at"] - NOTEBOOK_STARTED_AT, 2) if ANSWERS["task_3"]["answered_at"] else None,
    "task_3_elapsed_seconds": round(ANSWERS["task_3"]["answered_at"] - task_3_start, 2) if ANSWERS["task_3"]["answered_at"] else None,
}

timing.update(get_attempt_stats("task_1"))
timing.update(get_attempt_stats("task_2"))
timing.update(get_attempt_stats("task_3"))


# ---------------------------------------------------------------------
# Displayed option order
# ---------------------------------------------------------------------

displayed_order = []

for task_name, table in OPTIONS.items():
    for _, row in table[["run_id", "display_order"]].iterrows():
        displayed_order.append({
            "task_name": task_name,
            "run_id": int(row["run_id"]),
            "display_order": int(row["display_order"]),
        })

for _, row in WHAT_IF_OPTIONS[["run_id", "display_order"]].iterrows():
    displayed_order.append({
        "task_name": "task_3",
        "base_run_id": int(WHAT_IF_BASE_RUN_ID),
        "run_id": int(row["run_id"]),
        "display_order": int(row["display_order"]),
    })


# ---------------------------------------------------------------------
# Hidden Task 3 diagnostics
# ---------------------------------------------------------------------

task_3_diagnostic_cols = [
    "run_id",
    "display_order",
    "candidate_type",
    "valid_what_if",
    "same_performance_ok",
    "review_budget_10pct_lower_ok",
    "different_from_base_ok",
    "quality_drop_from_base",
    "review_budget_reduction_absolute",
    "review_budget_reduction_percent",
    "base_run_id",
    "base_quality_score",
    "base_review_budget",
    "minimum_acceptable_quality",
    "required_max_review_budget",
]

task_3_diagnostic_cols = [
    col for col in task_3_diagnostic_cols
    if col in WHAT_IF_OPTIONS.columns
]

task_3_candidate_diagnostics = WHAT_IF_OPTIONS[
    task_3_diagnostic_cols
].to_dict(orient="records")


# ---------------------------------------------------------------------
# Check history
# ---------------------------------------------------------------------

CHECK_HISTORY.append({
    "check_run": CHECK_RUN_COUNT,
    "checked_at_relative_seconds": round(CHECKED_AT - NOTEBOOK_STARTED_AT, 2),
    "valid_all": bool(results["valid_selection"].all()),
    "answers_snapshot": to_jsonable(ANSWERS),
    "records": to_jsonable(records),
})


# ---------------------------------------------------------------------
# Save JSON and CSV
# ---------------------------------------------------------------------

payload = {
    "participant_id": str(PARTICIPANT_ID).strip(),
    "questionnaire_id": str(QUESTIONNAIRE_ID).strip(),
    "session_id": SESSION_ID_FINAL,
    "study_condition": STUDY_CONDITION,
    "notebook_version": NOTEBOOK_VERSION,
    "study_random_seed": STUDY_RANDOM_SEED,
    "what_if_base_random_seed": WHAT_IF_BASE_RANDOM_SEED,

    "timing": timing,
    "checker_runs_in_session": CHECK_RUN_COUNT,

    "attempt_log": to_jsonable(ATTEMPT_LOG),
    "check_history": to_jsonable(CHECK_HISTORY),

    "displayed_order": displayed_order,
    "task_3_base_configuration": to_jsonable(
        WHAT_IF_BASE_ROW[RAW_DISPLAY_COLUMNS].to_dict()
    ),
    "task_3_candidate_diagnostics": to_jsonable(task_3_candidate_diagnostics),

    "answers": to_jsonable(records),
    "tasks": to_jsonable(TASKS),
    "what_if_task": to_jsonable(WHAT_IF_TASK),
}

json_path = OUTPUT_DIR / f"{participant_token}_{session_token}_condition_a_answers.json"
csv_path = OUTPUT_DIR / f"{participant_token}_{session_token}_condition_a_answers.csv"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

results.to_csv(csv_path, index=False)

print(f"Saved: {json_path}")
print(f"Saved: {csv_path}")
print(f"Questionnaire ID to use in the form: {QUESTIONNAIRE_ID}")

,task_name,display_order,run_id,quality_score,training_fraction,review_budget,corruption_level,shown_in_option_table,quality_rank_within_displayed_options,review_budget_rank_within_displayed_options,...,review_budget_margin_upper,quality_drop_from_base,review_budget_reduction_absolute,review_budget_reduction_percent,candidate_type,valid_what_if,valid_selection,n_violations,confidence,justification
0,task_1,6,1213,0.859425,0.55,0.375,none,True,9,10,...,0.0250,NaN,NaN,NaN,NaN,NaN,True,0,high,There are only 2 configurations that respect t...
1,task_2,9,1317,0.823985,0.32,0.250,none,True,9,1,...,0.0500,NaN,NaN,NaN,NaN,NaN,True,0,medium,valid soft constraints good quality
2,task_3,9,2939,0.882250,0.86,0.310,none,True,5,5,...,0.0275,-0.032409,0.065,0.173333,valid,True,True,0,medium,satisfies constraints


All selections satisfy the task constraints.
Saved: student_condition_a_outputs/P-F54434_20260505_200233_condition_a_answers.json
Saved: student_condition_a_outputs/P-F54434_20260505_200233_condition_a_answers.csv
Questionnaire ID to use in the form: P-F54434


## After the tasks

You are done with the notebook when the final cell saves the two output files. Please send these two files by email to sabri.manai@uj.edu.pl

Please now complete the short questionnaire provided by the facilitator.

Use the same `QUESTIONNAIRE_ID` shown by the notebook. This allows the notebook results and questionnaire answers to be linked without using your full name.

## Troubleshooting

- If the install cell fails, tell the facilitator.
- If the data-loading cell fails, do not continue.
- If a task table does not appear, rerun the setup cells from the top.
- If the final checker reports a violation, inspect the `*_ok` columns and revise only the relevant answer cell.